<a href="https://colab.research.google.com/github/Safin44/my-flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

**Lane:** Search performance / CTR opportunity ranking  
**Decision snapshot:** March 2026; one pseudonymized client × content item.  
**Purpose:** create a transparent, reproducible rule baseline that Week 5 can try to beat.

This notebook uses only observed March features. It does **not** use April outcomes, labels, product flags, URLs, titles, raw queries, or IDs as scoring features. Hash IDs are retained only as opaque row keys for joins and reproducibility.

**Access requirement:** request access to `FlyRank/internship-warehouse`, accept its data-use terms, and add a Hugging Face **Read** token as Colab Secret `HF_TOKEN` (or environment variable). Do not paste a token into the notebook or commit it. The notebook intentionally computes all counts and verdicts at runtime; no sample results are fabricated.

The notebook is designed to run from the repository root or in Colab. Output CSV is regenerated at `work/outputs/baseline_action_score.csv` and should remain untracked per the assignment.

## 1. Two signal checks and the one rule

### Signal A — CTR versus position (flag-linked)
The session's CTR-fix logic depends on whether a page's observed CTR is weak relative to its search position. I will bucket rows by March mean position and compare each bucket's median CTR. **CONFIRMED** means median CTR generally decreases as position gets worse; **OPPOSITE** means the reverse; **MIXED** means no clear monotonic direction; **FALSE** means the required data or a usable comparison is absent. The bucket table prints `n` so small buckets are visible.

### Signal B — search volume (quick-win flag-linked)
I will bucket March impressions into quartiles and compare median impressions and median clicks. **CONFIRMED** means higher impression buckets also show higher median clicks; **OPPOSITE** means not; **MIXED** means the evidence is inconsistent; **FALSE** means no usable variation or missing data. This is a signal check, not proof that a page is a quick win.

### Single baseline rule
Score a page using **high impressions × CTR shortfall relative to its position bucket**. Within each position bucket, compare the page's CTR to the bucket median. A positive shortfall means the page is below the peer median; the score is `log1p(impressions) × positive CTR shortfall`. Emit exactly one reason code, `CTR_GAP_AT_POSITION`, and one action label, `REVIEW_TITLE_SNIPPET`. This is a review-priority heuristic, not a causal claim or prediction of search ranking.

In [1]:
# Setup and safe warehouse access. No token is stored in this notebook.
%pip -q install duckdb pandas numpy huggingface_hub

import os
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Request access to FlyRank/internship-warehouse, "
        "accept its data terms, then add a Hugging Face READ token as Colab Secret HF_TOKEN "
        "or environment variable HF_TOKEN. Never hard-code or commit the token."
    )

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{safe_token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"
FEATURE_START = "2026-03-01"
FEATURE_END = "2026-04-01"

schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{FACT}', hive_partitioning=true) LIMIT 0"
).df()
columns = {str(c).lower() for c in schema["column_name"]}

def pick(candidates, required=True):
    for candidate in candidates:
        if candidate.lower() in columns:
            return candidate.lower()
    if required:
        raise KeyError(
            f"Required field not found; tried {candidates}. "
            f"Available fields: {sorted(columns)}"
        )
    return None

date_col = pick(["report_date", "date"])
client_col = pick(["client_hash_id"])
content_col = pick(["content_hash_id"])
impressions_col = pick(["gsc_impressions", "impressions"])
clicks_col = pick(["gsc_clicks", "clicks"])
position_col = pick(["gsc_avg_position", "gsc_position", "average_position", "position"])
availability_col = pick([
    "gsc_available", "gsc_is_available", "has_gsc_data",
    "search_console_available", "is_gsc_available", "gsc_data_available"
])

print("Schema fields detected:", len(columns))
print("Decision window:", FEATURE_START, "through", "2026-03-31")
print("Detected columns:", {
    "date": date_col, "client_key": client_col, "content_key": content_col,
    "impressions": impressions_col, "clicks": clicks_col,
    "position": position_col, "availability_boolean": availability_col
})

Schema fields detected: 31
Decision window: 2026-03-01 through 2026-03-31
Detected columns: {'date': 'report_date', 'client_key': 'client_hash_id', 'content_key': 'content_hash_id', 'impressions': 'gsc_impressions', 'clicks': 'gsc_clicks', 'position': 'gsc_avg_position', 'availability_boolean': 'gsc_data_available'}


## 2. Build the March-only feature frame and ranked queue

The grain is one client × content item, aggregated over March 2026 with the dataset's explicit Search Console availability boolean. The score uses only March impressions, clicks-derived CTR, and March mean position. IDs are keys only. The queue is written by this notebook; it is intentionally not committed as a data file.

In [2]:
# Aggregate March data only. No April/future columns or labels are selected.
def qi(identifier):
    # Identifiers are selected from the actual schema and quoted defensively.
    return '"' + identifier.replace('"', '""') + '"'

sql = f"""
SELECT
  {qi(client_col)} AS client_key,
  {qi(content_col)} AS content_key,
  SUM(COALESCE(TRY_CAST({qi(impressions_col)} AS DOUBLE), 0)) AS impressions,
  SUM(COALESCE(TRY_CAST({qi(clicks_col)} AS DOUBLE), 0)) AS clicks,
  AVG(TRY_CAST({qi(position_col)} AS DOUBLE)) AS mean_position
FROM read_parquet('{FACT}', hive_partitioning=true)
WHERE CAST({qi(date_col)} AS DATE) >= DATE '{FEATURE_START}'
  AND CAST({qi(date_col)} AS DATE) < DATE '{FEATURE_END}'
  AND {qi(availability_col)} IS TRUE
GROUP BY 1, 2
"""
features = con.execute(sql).df()
features["impressions"] = pd.to_numeric(features["impressions"], errors="coerce")
features["clicks"] = pd.to_numeric(features["clicks"], errors="coerce")
features["mean_position"] = pd.to_numeric(features["mean_position"], errors="coerce")
features = features.replace([np.inf, -np.inf], np.nan)
features = features[
    features["impressions"].gt(0) &
    features["clicks"].ge(0) &
    features["mean_position"].gt(0)
].copy()
features["ctr"] = features["clicks"] / features["impressions"]
features = features[features["ctr"].between(0, 1, inclusive="both")].copy()

if features.empty:
    raise RuntimeError("No usable March rows after the explicit availability and quality filters.")

# Signal A: position buckets and median CTR. Rank-based bins avoid assuming fixed position cutoffs.
features["position_bucket"] = pd.qcut(
    features["mean_position"].rank(method="first"),
    q=min(5, features["mean_position"].nunique()),
    labels=False,
    duplicates="drop"
)
position_table = (
    features.groupby("position_bucket", observed=True)
    .agg(n=("ctr", "size"),
         median_position=("mean_position", "median"),
         median_ctr=("ctr", "median"),
         median_impressions=("impressions", "median"))
    .reset_index()
)
display(position_table)
ctr_vals = position_table.sort_values("median_position")["median_ctr"].dropna().to_numpy()
if len(ctr_vals) < 3 or np.nanmax(ctr_vals) == np.nanmin(ctr_vals):
    ctr_verdict = "FALSE"
else:
    diffs = np.diff(ctr_vals)
    if np.all(diffs <= 0) and np.any(diffs < 0):
        ctr_verdict = "CONFIRMED"
    elif np.all(diffs >= 0) and np.any(diffs > 0):
        ctr_verdict = "OPPOSITE"
    else:
        ctr_verdict = "MIXED"
print(f"Signal A verdict: {ctr_verdict} — CTR by position bucket; bucket n is shown above.")

# Signal B: impressions quartiles versus clicks. Each bucket reports n.
features["volume_bucket"] = pd.qcut(
    features["impressions"].rank(method="first"),
    q=min(4, features["impressions"].nunique()),
    labels=False,
    duplicates="drop"
)
volume_table = (
    features.groupby("volume_bucket", observed=True)
    .agg(n=("clicks", "size"),
         median_impressions=("impressions", "median"),
         median_clicks=("clicks", "median"),
         median_ctr=("ctr", "median"))
    .reset_index()
)
display(volume_table)
click_vals = volume_table.sort_values("median_impressions")["median_clicks"].dropna().to_numpy()
if len(click_vals) < 3 or np.nanmax(click_vals) == np.nanmin(click_vals):
    volume_verdict = "FALSE"
else:
    diffs = np.diff(click_vals)
    if np.all(diffs >= 0) and np.any(diffs > 0):
        volume_verdict = "CONFIRMED"
    elif np.all(diffs <= 0) and np.any(diffs < 0):
        volume_verdict = "OPPOSITE"
    else:
        volume_verdict = "MIXED"
print(f"Signal B verdict: {volume_verdict} — clicks by impression bucket; bucket n is shown above.")

# One transparent rule: peer-median CTR shortfall weighted by available search volume.
peer_median = features.groupby("position_bucket", observed=True)["ctr"].transform("median")
features["ctr_shortfall"] = (peer_median - features["ctr"]).clip(lower=0)
features["score"] = np.log1p(features["impressions"]) * features["ctr_shortfall"]
features["reason_code"] = np.where(features["ctr_shortfall"] > 0, "CTR_GAP_AT_POSITION", "CTR_GAP_AT_POSITION")
features["action"] = "REVIEW_TITLE_SNIPPET"
features = features.sort_values(
    ["score", "impressions", "clicks"], ascending=[False, False, False]
).reset_index(drop=True)
features.insert(0, "rank", np.arange(1, len(features) + 1))

queue = features[[
    "rank", "client_key", "content_key", "score", "reason_code", "action",
    "impressions", "clicks", "ctr", "mean_position", "ctr_shortfall"
]].copy()

# The assignment specifies this output path. Create the directory from the repo root.
out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_path, index=False)
print(f"Rows scored: {len(queue):,}")
print(f"Wrote ranked queue: {out_path.as_posix()}")
display(queue.head(10).drop(columns=["client_key", "content_key"]))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,median_position,median_ctr,median_impressions
0,0,35061,3.152826,0.000719,396.0
1,1,35061,5.750000,0.000000,165.0
2,2,35060,8.612729,0.000000,173.0
3,3,35061,16.830556,0.000000,258.0
4,4,35061,40.279661,0.000000,97.0


Signal A verdict: CONFIRMED — CTR by position bucket; bucket n is shown above.


,volume_bucket,n,median_impressions,median_clicks,median_ctr
0,0,43826,4.0,0.0,0.000000
1,1,43826,69.0,0.0,0.000000
2,2,43826,427.0,0.0,0.000000
3,3,43826,3041.5,6.0,0.001949


Signal B verdict: CONFIRMED — clicks by impression bucket; bucket n is shown above.
Rows scored: 175,304
Wrote ranked queue: work/outputs/baseline_action_score.csv


,rank,score,reason_code,action,impressions,clicks,ctr,mean_position,ctr_shortfall
0,1,0.007304,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,38000.0,1.0,0.000026,2.740744,0.000693
1,2,0.007277,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,24908.0,0.0,0.000000,4.009764,0.000719
2,3,0.006904,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,14813.0,0.0,0.000000,3.426946,0.000719
3,4,0.006852,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,48049.0,4.0,0.000083,4.406859,0.000636
4,5,0.006842,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,24259.0,1.0,0.000041,2.784944,0.000678
5,6,0.006787,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,12588.0,0.0,0.000000,1.902457,0.000719
6,7,0.006702,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,11187.0,0.0,0.000000,4.376694,0.000719
7,8,0.006682,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,10886.0,0.0,0.000000,4.204145,0.000719
8,9,0.006654,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,10462.0,0.0,0.000000,4.259897,0.000719
9,10,0.006613,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,9887.0,0.0,0.000000,1.953104,0.000719


## 3. Top-10 skeptical review

Each row gets an action, the single reason code, a confidence note grounded in the observed rule inputs, and a specific condition that could make the recommendation wrong. These are hypotheses for human review, not asserted page defects. Pseudonymous keys are omitted from the displayed review.

In [3]:
# Exactly ten rows are reviewed when the queue has at least ten eligible items.
if len(queue) < 10:
    print(f"Only {len(queue)} eligible rows are available; reviewing all of them rather than inventing rows.")
review = queue.head(10).copy()
review["confidence_note"] = np.select(
    [
        review["ctr_shortfall"] > 0.02,
        review["ctr_shortfall"] > 0
    ],
    [
        "Larger CTR gap versus the position-bucket median; still observational.",
        "Positive but modest CTR gap versus the position-bucket median."
    ],
    default="Weak/zero peer-median gap; rank is likely driven by tie-breaks."
)
review["what_would_make_it_wrong"] = (
    "The position bucket may mix different intents/SERP features; the low CTR may be expected, "
    "or title/snippet changes may not be available or appropriate."
)
review["action"] = "REVIEW_TITLE_SNIPPET"
review["reason_code"] = "CTR_GAP_AT_POSITION"
review_display = review[[
    "rank", "action", "reason_code", "score", "impressions", "ctr",
    "mean_position", "confidence_note", "what_would_make_it_wrong"
]]
display(review_display)
print("Top-10 review rows shown:", len(review_display))

,rank,action,reason_code,score,impressions,ctr,mean_position,confidence_note,what_would_make_it_wrong
0,1,REVIEW_TITLE_SNIPPET,CTR_GAP_AT_POSITION,0.007304,38000.0,0.000026,2.740744,Positive but modest CTR gap versus the positio...,The position bucket may mix different intents/...
1,2,REVIEW_TITLE_SNIPPET,CTR_GAP_AT_POSITION,0.007277,24908.0,0.000000,4.009764,Positive but modest CTR gap versus the positio...,The position bucket may mix different intents/...
2,3,REVIEW_TITLE_SNIPPET,CTR_GAP_AT_POSITION,0.006904,14813.0,0.000000,3.426946,Positive but modest CTR gap versus the positio...,The position bucket may mix different intents/...
3,4,REVIEW_TITLE_SNIPPET,CTR_GAP_AT_POSITION,0.006852,48049.0,0.000083,4.406859,Positive but modest CTR gap versus the positio...,The position bucket may mix different intents/...
4,5,REVIEW_TITLE_SNIPPET,CTR_GAP_AT_POSITION,0.006842,24259.0,0.000041,2.784944,Positive but modest CTR gap versus the positio...,The position bucket may mix different intents/...
5,6,REVIEW_TITLE_SNIPPET,CTR_GAP_AT_POSITION,0.006787,12588.0,0.000000,1.902457,Positive but modest CTR gap versus the positio...,The position bucket may mix different intents/...
6,7,REVIEW_TITLE_SNIPPET,CTR_GAP_AT_POSITION,0.006702,11187.0,0.000000,4.376694,Positive but modest CTR gap versus the positio...,The position bucket may mix different intents/...
7,8,REVIEW_TITLE_SNIPPET,CTR_GAP_AT_POSITION,0.006682,10886.0,0.000000,4.204145,Positive but modest CTR gap versus the positio...,The position bucket may mix different intents/...
8,9,REVIEW_TITLE_SNIPPET,CTR_GAP_AT_POSITION,0.006654,10462.0,0.000000,4.259897,Positive but modest CTR gap versus the positio...,The position bucket may mix different intents/...
9,10,REVIEW_TITLE_SNIPPET,CTR_GAP_AT_POSITION,0.006613,9887.0,0.000000,1.953104,Positive but modest CTR gap versus the positio...,The position bucket may mix different intents/...


Top-10 review rows shown: 10


## 4. Weak picks and leakage check

The rule can elevate a page because of a noisy peer median, a tiny position bucket, mixed search intent, SERP features, brand demand, or data-sync issues. Review the weakest positive-score picks rather than assuming every score is a true content opportunity. The queue's bottom rows have little or no CTR shortfall and are weak picks by construction.

Leakage guard: score inputs are March impressions, March clicks-derived CTR, and March mean position. April outcome data, decline labels, product flags, IDs-as-features, URLs, titles, and queries are not used. Hash keys are only retained as output identifiers.

In [4]:
# Show five weakest picks among the positive-score rows (or bottom of queue if fewer).
positive = queue[queue["score"] > 0]
weak = (positive.tail(5) if len(positive) else queue.tail(5)).copy()
weak["why_weak"] = np.where(
    weak["ctr_shortfall"] <= 0,
    "No positive CTR gap versus position peers; low-priority/tie-break row.",
    "Small score relative to the queue; the gap may be noise or peer-bucket artefact."
)
display(weak[[
    "rank", "score", "reason_code", "action", "impressions", "ctr",
    "mean_position", "why_weak"
]].sort_values("rank"))

# Structural assertions: these are the only score inputs and the decision window is March.
score_inputs = {"impressions", "clicks", "ctr", "mean_position", "ctr_shortfall", "position_bucket"}
forbidden_tokens = ("label", "future", "april", "outcome", "product_flag", "decline")
assert not any(any(token in col.lower() for token in forbidden_tokens)
               for col in score_inputs), "Potential leakage token found in scoring inputs."
assert set(queue["reason_code"].unique()) == {"CTR_GAP_AT_POSITION"}
assert set(queue["action"].unique()) == {"REVIEW_TITLE_SNIPPET"}
assert out_path.exists() and out_path.stat().st_size > 0
print("Leakage self-check: PASS — March-only observed signals; no labels/future-window inputs.")
print("One reason code:", queue["reason_code"].unique().tolist())
print("One action label:", queue["action"].unique().tolist())
print("Output exists:", out_path.as_posix())

,rank,score,reason_code,action,impressions,ctr,mean_position,why_weak
17525,17526,0.000011,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,1394.0,0.000717,4.108362,Small score relative to the queue; the gap may...
17526,17527,0.000011,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,5574.0,0.000718,3.530349,Small score relative to the queue; the gap may...
17527,17528,0.000007,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,1393.0,0.000718,1.390958,Small score relative to the queue; the gap may...
17528,17529,0.000004,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,1392.0,0.000718,3.063184,Small score relative to the queue; the gap may...
17529,17530,0.000002,CTR_GAP_AT_POSITION,REVIEW_TITLE_SNIPPET,8348.0,0.000719,4.212587,Small score relative to the queue; the gap may...


Leakage self-check: PASS — March-only observed signals; no labels/future-window inputs.
One reason code: ['CTR_GAP_AT_POSITION']
One action label: ['REVIEW_TITLE_SNIPPET']
Output exists: work/outputs/baseline_action_score.csv


## Self-check

- [x] Two signal checks implemented with visible bucket tables and per-bucket `n`; verdicts are computed from observed data at runtime (at least one check is linked to the session's CTR-fix logic).
- [x] One transparent score, one reason code (`CTR_GAP_AT_POSITION`), and one action (`REVIEW_TITLE_SNIPPET`).
- [x] Notebook writes `work/outputs/baseline_action_score.csv`.
- [x] Top-10 review includes action, reason, confidence, and what could make each pick wrong.
- [x] Weak picks and leakage guard included; no label-derived or future-window inputs are used.
- [ ] Run all cells in the authorized data environment and inspect the actual bucket tables/verdicts and top ten.
- [ ] Commit this notebook at `work/notebooks/w04_baseline_score.ipynb`; keep the generated CSV out of git.
- [ ] Commit any required run receipts/metrics JSONs, then submit your repository URL.

**Important:** this notebook has not been run against the gated FlyRank warehouse in this environment. Do not claim the data-derived verdicts or queue are verified until you run it with authorized `HF_TOKEN` access.